# ✅ LangGraph Compliance Checklist Flow

## What We're Building

A compliance review workflow that:
1. **Defines requirements** — loads a compliance framework checklist
2. **Gathers evidence** — local tools extract evidence from documents
3. **Checks compliance** — LLM evaluates each requirement against evidence
4. **Generates report** — produces a structured compliance checklist report

## Architecture

```
Compliance Framework
       │
       ▼
┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│  Define      │ ──▶│  Gather      │ ──▶│  Check       │ ──▶│  Generate    │
│  Requirements│    │  Evidence     │    │  Compliance  │    │  Report      │
└──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
```

## Stack
- **LangGraph** — orchestrates the compliance review pipeline
- **Local tools** — Python functions to simulate document evidence extraction
- **Ollama** — LLM for compliance assessment and report generation

## Step 1 — Imports & Configuration

In [ ]:
import json
from typing import TypedDict
from datetime import datetime
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:4b", temperature=0.1)
print("All imports successful!")

## Step 2 — Compliance Framework & Simulated Documents

We define a simplified compliance checklist (inspired by SOC 2 / ISO 27001)
and simulate company documents that contain (or lack) the required evidence.

In [ ]:
COMPLIANCE_REQUIREMENTS = [
    {
        "id": "AC-01",
        "domain": "Access Control",
        "requirement": "Multi-factor authentication (MFA) must be enabled for all admin accounts",
        "evidence_needed": "MFA configuration records, admin account audit"
    },
    {
        "id": "AC-02",
        "domain": "Access Control",
        "requirement": "Access reviews must be conducted quarterly for privileged accounts",
        "evidence_needed": "Access review logs, review schedule"
    },
    {
        "id": "DP-01",
        "domain": "Data Protection",
        "requirement": "All data at rest must be encrypted using AES-256 or equivalent",
        "evidence_needed": "Encryption configuration, key management policy"
    },
    {
        "id": "DP-02",
        "domain": "Data Protection",
        "requirement": "Backup recovery tests must be performed at least annually",
        "evidence_needed": "Backup test reports, recovery time records"
    },
    {
        "id": "IR-01",
        "domain": "Incident Response",
        "requirement": "An incident response plan must be documented and tested annually",
        "evidence_needed": "IR plan document, tabletop exercise records"
    },
    {
        "id": "CM-01",
        "domain": "Change Management",
        "requirement": "All production changes must go through a documented approval process",
        "evidence_needed": "Change request logs, approval records"
    },
]

# Simulated company documents (some have evidence, some don't)
COMPANY_DOCUMENTS = {
    "security_config.md": (
        "## Security Configuration\n"
        "All admin accounts have MFA enabled via Okta since January 2025.\n"
        "MFA enforcement policy was reviewed on 2026-01-15.\n"
        "Database encryption uses AES-256 via AWS KMS.\n"
        "S3 buckets have server-side encryption enabled (SSE-S3).\n"
        "Key rotation is scheduled every 90 days."
    ),
    "access_review_q1_2026.md": (
        "## Q1 2026 Access Review\n"
        "Reviewed: 2026-03-15\n"
        "Reviewer: Security Team Lead\n"
        "- 12 admin accounts reviewed\n"
        "- 2 accounts deactivated (former employees)\n"
        "- All remaining accounts confirmed active and justified\n"
        "Next review scheduled: 2026-06-15"
    ),
    "incident_response_plan.md": (
        "## Incident Response Plan v3.1\n"
        "Last updated: 2025-11-01\n"
        "Tabletop exercise conducted: 2025-10-15\n"
        "Next exercise scheduled: 2026-10-15\n"
        "Covers: detection, containment, eradication, recovery, lessons learned"
    ),
    "change_management_policy.md": (
        "## Change Management Policy\n"
        "All production changes require a JIRA ticket with:\n"
        "- Description of change\n"
        "- Risk assessment\n"
        "- Approval from team lead + security\n"
        "Emergency changes require post-hoc review within 48 hours."
    ),
    # NOTE: No backup test report document exists — this is a gap
}

print(f"Loaded {len(COMPLIANCE_REQUIREMENTS)} requirements")
print(f"Available documents: {len(COMPANY_DOCUMENTS)}")
for doc in COMPANY_DOCUMENTS:
    print(f"  📄 {doc}")

## Step 3 — Define State & Local Evidence Tool

In [ ]:
class ComplianceState(TypedDict):
    requirements_json: str      # Compliance requirements as JSON
    evidence_report: str        # Gathered evidence from documents
    compliance_assessment: str  # LLM assessment per requirement
    final_checklist: str        # Final formatted checklist report


def search_evidence(requirement: dict, documents: dict) -> str:
    """Local tool: search documents for evidence matching a requirement."""
    keywords = requirement["requirement"].lower().split()
    # Keep meaningful keywords
    stop_words = {"must", "be", "all", "for", "the", "an", "a", "or", "and", "at", "least"}
    keywords = [w for w in keywords if w not in stop_words and len(w) > 3]

    found = []
    for doc_name, content in documents.items():
        content_lower = content.lower()
        matches = [kw for kw in keywords if kw in content_lower]
        if len(matches) >= 2:  # At least 2 keyword matches
            found.append(f"📄 {doc_name}: {content[:200]}...")

    if found:
        return "\n".join(found)
    return "❌ No matching evidence found in available documents."


print("State and evidence search tool defined")

## Step 4 — Define Graph Nodes

| Node | Role |
|------|------|
| `define_requirements` | Load and validate the compliance checklist |
| `gather_evidence` | Search documents for evidence per requirement |
| `check_compliance` | LLM assesses compliance for each requirement |
| `generate_report` | LLM produces the final compliance checklist report |

In [ ]:
def define_requirements(state: ComplianceState) -> dict:
    """Load and validate compliance requirements."""
    print("📋 Node: define_requirements — Loading checklist...")
    reqs = json.loads(state["requirements_json"])
    domains = set(r["domain"] for r in reqs)
    print(f"     {len(reqs)} requirements across {len(domains)} domains: {', '.join(domains)}")
    return {}


def gather_evidence(state: ComplianceState) -> dict:
    """Search documents for evidence matching each requirement."""
    print("🔍 Node: gather_evidence — Searching documents...")
    reqs = json.loads(state["requirements_json"])
    evidence_lines = []

    for req in reqs:
        evidence = search_evidence(req, COMPANY_DOCUMENTS)
        has_evidence = "No matching evidence" not in evidence
        status = "✓ Found" if has_evidence else "✗ Missing"
        print(f"     {req['id']}: {status}")
        evidence_lines.append(
            f"Requirement {req['id']} ({req['domain']}): {req['requirement']}\n"
            f"Evidence needed: {req['evidence_needed']}\n"
            f"Evidence found:\n{evidence}\n"
        )

    return {"evidence_report": "\n---\n".join(evidence_lines)}


assess_prompt = ChatPromptTemplate.from_template(
    """You are a compliance auditor. Assess each requirement based on
the evidence gathered.

Evidence Report:
{evidence_report}

For EACH requirement, provide:
- REQUIREMENT ID and name
- STATUS: COMPLIANT / PARTIALLY COMPLIANT / NON-COMPLIANT / UNABLE TO ASSESS
- EVIDENCE SUMMARY: What evidence supports or contradicts compliance
- GAPS: What's missing (if anything)
- RISK LEVEL: Low / Medium / High (if non-compliant)

Be objective and evidence-based. Do not assume compliance without evidence."""
)


def check_compliance(state: ComplianceState) -> dict:
    """LLM assesses compliance for each requirement."""
    print("⚖️ Node: check_compliance — Assessing...")
    chain = assess_prompt | llm | StrOutputParser()
    assessment = chain.invoke({"evidence_report": state["evidence_report"]})
    return {"compliance_assessment": assessment}


report_prompt = ChatPromptTemplate.from_template(
    """You are a compliance officer. Generate a final compliance checklist report.

Assessment:
{assessment}

Requirements:
{requirements}

Generate a professional compliance report with:
1. EXECUTIVE SUMMARY — overall compliance posture (X of Y requirements met)
2. CHECKLIST TABLE — each requirement with status (✅ / ⚠️ / ❌)
3. CRITICAL GAPS — non-compliant items with recommended remediation
4. TIMELINE — suggested remediation deadlines
5. SIGN-OFF — report date and next review date

Today's date: {today}"""
)


def generate_report(state: ComplianceState) -> dict:
    """Generate the final compliance checklist report."""
    print("📝 Node: generate_report — Writing final checklist...")
    chain = report_prompt | llm | StrOutputParser()
    report = chain.invoke({
        "assessment": state["compliance_assessment"],
        "requirements": state["requirements_json"],
        "today": datetime.now().strftime("%Y-%m-%d"),
    })
    return {"final_checklist": report}


print("All nodes defined")

## Step 5 — Build & Compile the Graph

In [ ]:
workflow = StateGraph(ComplianceState)

workflow.add_node("define_requirements", define_requirements)
workflow.add_node("gather_evidence", gather_evidence)
workflow.add_node("check_compliance", check_compliance)
workflow.add_node("generate_report", generate_report)

workflow.set_entry_point("define_requirements")
workflow.add_edge("define_requirements", "gather_evidence")
workflow.add_edge("gather_evidence", "check_compliance")
workflow.add_edge("check_compliance", "generate_report")
workflow.add_edge("generate_report", END)

app = workflow.compile()
print("Compliance checklist workflow compiled")

## Step 6 — Run the Full Compliance Review

In [ ]:
result = app.invoke({
    "requirements_json": json.dumps(COMPLIANCE_REQUIREMENTS),
    "evidence_report": "",
    "compliance_assessment": "",
    "final_checklist": "",
})

print("\n" + "="*60)
print("🔍 EVIDENCE REPORT")
print("="*60)
print(result["evidence_report"])

print("\n" + "="*60)
print("⚖️ COMPLIANCE ASSESSMENT")
print("="*60)
print(result["compliance_assessment"])

print("\n" + "="*60)
print("📋 FINAL COMPLIANCE CHECKLIST REPORT")
print("="*60)
print(result["final_checklist"])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Local evidence tools** | Python functions search documents — no external APIs |
| **Structured requirements** | JSON-based checklist is extensible and auditable |
| **LLM assessment** | Ollama evaluates compliance based on gathered evidence |
| **Gap detection** | Missing evidence surfaces non-compliant items |
| **Report generation** | Final output is a professional compliance checklist |

## Why LangGraph for Compliance?

- Compliance reviews are naturally sequential and structured
- Each node has clear inputs/outputs — easy to audit
- Local tools keep sensitive document processing in-house
- LLM adds judgment (assessing adequacy of evidence) without replacing human review

## 🔧 Extensions

- **Real document parsing**: Integrate PDF/Word extraction with PyMuPDF or python-docx
- **HITL approval**: Add `interrupt_before` generate_report for auditor sign-off
- **Framework templates**: Load different standards (SOC 2, ISO 27001, HIPAA)
- **Continuous monitoring**: Schedule periodic re-runs and track compliance trends
- **Evidence linking**: Attach specific document excerpts to each requirement